# Subset data creation

### Inpecting Raw data 

In [6]:
import json
import random
import pandas as pd
import re

In [ ]:
# Path to one of your files
file_path = 'data/electronics.json/Electronics_5.json'
file_path2 = 'data\home.json\Home_and_Kitchen_5.json'
file_path3 = 'data\sports.json\Sports_and_Outdoors_5.json'
with open(file_path3, 'r', encoding='utf-8') as f:
    first_line = f.readline()
    first_record = json.loads(first_line)
    
# Print the structur
print(json.dumps(first_record, indent=4))

{
    "reviewerID": "AIXZKN4ACSKI",
    "asin": "1881509818",
    "reviewerName": "David Briner",
    "helpful": [
        0,
        0
    ],
    "reviewText": "This came in on time and I am veru happy with it, I haved used it already and it makes taking out the pins in my glock 32 very easy",
    "overall": 5.0,
    "summary": "Woks very good",
    "unixReviewTime": 1390694400,
    "reviewTime": "01 26, 2014"
}


### Raw data to subset creation

In [ ]:
file_paths = {
    "Electronics": "data/electronics.json/Electronics_5.json",
    "Home_and_Kitchen": "data/home.json/Home_and_Kitchen_5.json",
    "Sports_and_Outdoors": "data/sports.json/Sports_and_Outdoors_5.json"
}

# 12,000 per category (Total: 36,000)
SAMPLES_PER_CATEGORY = 12000
full_subset = []

def load_and_sample(path, category_name, n):
    category_data = []
    print(f"Loading all fields from {category_name}...")
    
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                # Load the full JSON object
                record = json.loads(line)
                # Inject a category label so you know where it came from later
                record["category_label"] = category_name 
                category_data.append(record)
            except json.JSONDecodeError:
                continue
    
    # Perform random sampling if we have enough data
    if len(category_data) >= n:
        return random.sample(category_data, n)
    else:
        print(f"Warning: {category_name} only has {len(category_data)} samples.")
        return category_data

# Execute for each file
for category, path in file_paths.items():
    subset = load_and_sample(path, category, SAMPLES_PER_CATEGORY)
    full_subset.extend(subset)

random.shuffle(full_subset)

# Save to a new JSON Lines file 
with open("subset.json", "w", encoding="utf-8") as f_out:
    for entry in full_subset:
        f_out.write(json.dumps(entry) + "\n")

print(f"\nSaved {len(full_subset)} complete records to 'subset.json'.")

Loading all fields from Electronics...
Loading all fields from Home_and_Kitchen...
Loading all fields from Sports_and_Outdoors...

Saved 36000 complete records to 'subset.json'.


# Data cleaning

### load from json and convert to df

In [9]:
import json
import pandas as pd

data = []
with open('data/subset.json', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue # Skip empty lines
        
        # Strip trailing commas if they exist from manual concatenation
        if line.endswith(','):
            line = line[:-1]
            
        try:
            data.append(json.loads(line))
        except json.JSONDecodeError:
            # If a line fails, skip it or check if it's the start/end of a list [ ]
            continue

df = pd.DataFrame(data)
print(f"Loaded {len(df)} rows successfully.")

Loaded 35999 rows successfully.


### map rating to numbers

In [ ]:
def map_sentiment(rating):
    if rating <= 2: return 0 # Negative
    if rating == 3: return 1 # Neutral
    return 2                 # Positive

df['sentiment'] = df['overall'].apply(map_sentiment)

### Text Cleaning Function

In [ ]:

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text) # Remove HTML tags
    text = re.sub(r'[^a-z0-9\s]', '', text) # Remove special chars but keep space
    return text

df['cleaned_review'] = df['reviewText'].apply(clean_text)


In [11]:
df.head()

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,category_label,sentiment,cleaned_review
0,A3NA3M6ZNS6LLZ,B007261A36,BookWorm620,"[126, 129]",This camera is not as good as some you can get...,5.0,Feature rich and takes beatiful Pics,1336089600,"05 4, 2012",Electronics,2,this camera is not as good as some you can get...
1,A3VPFM3KVTM8GZ,B007P4YAPK,The One,"[2, 3]",I got my TF300T last July with the matching ke...,2.0,3 exchanges in the first 7 months!!!,1368662400,"05 16, 2013",Electronics,0,i got my tf300t last july with the matching ke...
2,AO6URI25USNBS,B00457SRBI,"Turmatic ""Turmat""","[4, 7]",I have tried three different red dot site for ...,5.0,This is the one that worked,1365379200,"04 8, 2013",Sports_and_Outdoors,2,i have tried three different red dot site for ...
3,A2Q0ATLV3G7H5B,B003OCQAAU,stacy l.,"[0, 0]",These sheets were okay at first. I actually l...,3.0,Okay at first,1387238400,"12 17, 2013",Home_and_Kitchen,1,these sheets were okay at first i actually li...
4,AGFEGIMRTUOBP,B003YIFHJY,Marcello,"[0, 0]",I love this product especially coupled with a ...,5.0,This is simply a great product.,1360454400,"02 10, 2013",Electronics,2,i love this product especially coupled with a ...
